# Train Roman's LoRA on Kaggle (Z-Image Turbo, free)

Runs unattended with **Save Version → Save & Run All**. It either works end to end or stops
within about 20 minutes with the reason as the last line and in `status.json`.

- **Input:** the private dataset from `uv run ca lora prepare` (images + `.txt` captions that
  start with the trigger word). To continue an earlier run, also attach that version's output.
- **Settings:** Accelerator **GPU T4 x2** (only GPU 0 is used), Internet **on**.
- **Pinned:** AI Toolkit at commit `f7a1fb9a…` (every config key checked against its code) and
  every package it needs, each at a version with a Python 3.13 wheel. Kaggle's torch stays.
  bitsandbytes is removed; the optimizer is PyTorch AdamW. Nothing runs in fp16: the model,
  text encoder, VAE, training and saved files are bf16 (emulated on the T4: slower, correct).
- **Smoke test first (≤ 15 min):** imports, AI Toolkit's own config loader, model on GPU 0,
  3 real steps with a loss check, a saved checkpoint reloaded (finite, learned, step stored for
  resume), 1 sample at 512 px checked. Only then the full run starts, in the same session.
- **Full run:** 1200 steps, buckets 512/768, a checkpoint + samples every 100 steps (last 4
  kept), a progress line every 20 steps (step, loss, s/step, ETA, free RAM, GPU memory, last
  checkpoint), the 0/NaN-loss guard, and a stop before the 9 h session limit. The next session
  resumes from the newest checkpoint that learned something.
- **Output (`/kaggle/working`):** `lora_output/<job>/` (checkpoints, samples), `train_log.txt`,
  `status.json`, `smoke_test/`, and one `<job>_outputs.zip`.

The photos stay in `/tmp` and are not part of the output.

In [ ]:
import time

NOTEBOOK_START = time.time()  # the session clock

# AI Toolkit at a fixed commit (never "latest"); every config key below was checked against
# its config code at this commit.
TOOLKIT_REPO = "https://github.com/ostris/ai-toolkit.git"
TOOLKIT_COMMIT = "f7a1fb9a77e5418269b1e9eda4ace6d7c1b7bb47"
BASE_MODEL = "Tongyi-MAI/Z-Image-Turbo"
ADAPTER_REPO = "ostris/zimage_turbo_training_adapter"
ADAPTER_FILE = "zimage_turbo_training_adapter_v2.safetensors"

# Training: AI Toolkit's example defaults, adjusted for one free T4.
STEPS = 1200
RANK = 16  # linear and linear_alpha: the example default
LEARNING_RATE = 1e-4
OPTIMIZER = "adamw"  # torch.optim.AdamW (bitsandbytes is not installed at all)
RESOLUTIONS = [512, 768]  # buckets; 1024 took ~45 s/step on the T4
FALLBACK_RESOLUTIONS = [512]  # once, if the GPU runs out of memory before a checkpoint
SAVE_EVERY = 100  # a checkpoint + samples every N steps
KEEP_CHECKPOINTS = 4  # only the last 4, to save disk
SAMPLE_SIZE = 768
SAMPLE_PROMPTS = [
    "[trigger], a man in a navy suit in a modern office, photorealistic portrait, window light",
    "[trigger], close-up portrait of a man, neutral grey background, studio light",
    "a man in a navy suit in a modern office, photorealistic portrait, window light",
]
GUARD_STEPS = 30  # stop if the loss is 0 / NaN / inf for this many first steps
PROGRESS_EVERY = 20  # a progress line every N steps
MIN_IMAGES = 9

# Time: everything must end before the session limit.
SESSION_HOURS = 9.0
END_MARGIN_MINUTES = 25  # kept free for stopping, packing and status.json
SMOKE_MINUTES = 15
SMOKE_STEPS = 3
SMOKE_RESOLUTION = 768  # the largest training size, so its memory peak is tested
SMOKE_SAMPLE_SIZE = 512

WORK = "/kaggle/working"
OUTPUT_DIR = f"{WORK}/lora_output"  # checkpoints and samples: saved as notebook output
SMOKE_DIR = f"{WORK}/smoke_test"
SCRATCH = "/tmp/ca_lora"  # dataset copy, toolkit and base models: not saved

# Packages: AI Toolkit's requirements at TOOLKIT_COMMIT (+ its "-r" include), each pinned to a
# version with a Python 3.13 manylinux wheel (checked on PyPI on 2026-10-10). Exceptions:
# lycoris-lora 1.8.3 is pure Python without a wheel (it installed from source on Kaggle; the
# "nearest" wheel is 3.3.0, a risky major jump); scipy 1.12.0 has no 3.13 wheel -> 1.14.1;
# opencv stays below 5 (5.0 removed APIs). Unpinned requirements: the newest version with a
# 3.13 wheel released on or before the commit date (2026-10-09).
SKIP_PACKAGES = {
    "torch", "torchvision", "torchaudio",  # Kaggle's CUDA build stays
    "bitsandbytes",  # only the 8-bit optimizers import it; it is uninstalled
    "torchcodec",  # audio only; built against one torch version
    "gradio", "fastapi", "uvicorn",  # UI and inference server, not training
}
PINNED: dict[str, str] = {
    "torchao": "torchao==0.10.0",
    "safetensors": "safetensors==0.8.0",
    "transformers": "transformers==5.5.3",
    "lycoris-lora": "lycoris-lora==1.8.3",
    "flatten-json": "flatten_json==0.1.14",
    "pyyaml": "pyyaml==6.0.3",
    "oyaml": "oyaml==1.0",
    "tensorboard": "tensorboard==2.21.0",
    "kornia": "kornia==0.8.3",
    "invisible-watermark": "invisible-watermark==0.2.0",
    "einops": "einops==0.8.2",
    "accelerate": "accelerate==1.15.0",
    "toml": "toml==0.10.2",
    "albumentations": "albumentations==1.4.15",
    "albucore": "albucore==0.0.16",
    "pydantic": "pydantic==2.14.0",
    "omegaconf": "omegaconf==2.4.0",
    "open-clip-torch": "open_clip_torch==3.3.0",
    "timm": "timm==1.0.22",
    "prodigyopt": "prodigyopt==1.1.2",
    "controlnet-aux": "controlnet_aux==0.0.10",
    "python-dotenv": "python-dotenv==1.2.4",
    "hf-transfer": "hf_transfer==0.1.9",
    "lpips": "lpips==0.1.4",
    "pytorch-fid": "pytorch_fid==0.3.0",
    "optimum-quanto": "optimum-quanto==0.2.4",
    "sentencepiece": "sentencepiece==0.2.2",
    "huggingface-hub": "huggingface_hub==1.23.0",
    "peft": "peft==0.18.1",
    "python-slugify": "python-slugify==9.1.3",
    "opencv-python": "opencv-python==4.14.0.94",
    "pytorch-wavelets": "pytorch-wavelets==1.3.0",
    "matplotlib": "matplotlib==3.10.1",
    "setuptools": "setuptools==84.0.0",
    "av": "av==16.0.1",
    "librosa": "librosa==0.11.0",
    "mutagen": "mutagen==1.47.0",
    "scipy": "scipy==1.14.1",
}
EXTRA_PINS = ["opencv-python-headless==4.14.0.94"]  # albumentations' opencv: also below 5

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from typing import NoReturn

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU only (T4 x2 has two); before torch
import torch  # noqa: E402 - after CUDA_VISIBLE_DEVICES

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SCRATCH, exist_ok=True)
os.environ["HF_HOME"] = f"{SCRATCH}/hf"  # base models stay out of /kaggle/working
# The trainer runs as a subprocess: one GPU, and a segfault prints the Python stack of every
# thread into its log.
TRAIN_ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "PYTHONFAULTHANDLER": "1"}
STATUS: dict[str, object] = {"status": "started"}


def save_status(**changes: object) -> None:
    STATUS.update(changes)
    Path(OUTPUT_DIR, "status.json").write_text(json.dumps(STATUS, indent=2))


def fail(reason: str) -> NoReturn:
    """Stop the notebook: the reason goes to status.json and is the last line printed."""
    save_status(status=f"failed: {reason}")
    raise SystemExit(f"FAILED: {reason}")


def sh(command: str) -> None:
    """A setup command; a failure stops the notebook with its reason."""
    print("$", command, flush=True)
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    if done.returncode != 0:
        fail(f"{command!r} failed: {(done.stdout + done.stderr).strip()[-600:]}")


if not torch.cuda.is_available():
    fail("no GPU: Settings -> Accelerator -> GPU T4 x2, then Save & Run All again")
if torch.cuda.device_count() != 1:
    fail(f"{torch.cuda.device_count()} GPUs visible; CUDA_VISIBLE_DEVICES=0 did not apply")
MAJOR, MINOR = torch.cuda.get_device_capability(0)


def bf16_works() -> bool:
    """bf16 matmul, conv and softmax on this GPU give finite results close to fp32."""
    try:
        torch.manual_seed(0)
        a = torch.randn(512, 512, device="cuda")
        image = torch.randn(1, 16, 64, 64, device="cuda")
        kernel = torch.randn(32, 16, 3, 3, device="cuda")
        product = (a.bfloat16() @ a.bfloat16()).float()
        conv = torch.nn.functional.conv2d(image.bfloat16(), kernel.bfloat16()).float()
        soft = torch.softmax(a.bfloat16() * 30, dim=-1).float()
        close = torch.allclose(product, a @ a, rtol=0.05, atol=2.0)
        return bool(torch.isfinite(product).all() and torch.isfinite(conv).all()
                    and torch.isfinite(soft).all() and close)
    except RuntimeError as error:
        print("bf16 test failed:", error)
        return False


# Never fp16 (Z-Image overflows it): bf16 is native from compute 8.0 and emulated on the
# T4/P100 (slower but correct). If bf16 does not work at all, nothing would train correctly.
if MAJOR < 8 and not bf16_works():
    fail("bf16 does not work on this GPU and fp16 overflows Z-Image")
DTYPE = "bf16"
GPU_NAME = torch.cuda.get_device_name(0)
print(f"GPU 0: {GPU_NAME} (compute {MAJOR}.{MINOR}), "
      f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB, dtype {DTYPE}"
      + ("" if MAJOR >= 8 else " (emulated on this GPU: slower)"))
save_status(dtype=DTYPE, gpu=GPU_NAME)

In [ ]:
# The dataset from `ca lora prepare`, found wherever Kaggle put it (unpacked or still zipped,
# any folder depth, any image-extension case). A missing or untidy caption is fixed: every
# caption starts with the trigger word, the most common first word of the captions.
import shutil
import zipfile
from collections import Counter

IMAGE_TYPES = {".jpg", ".jpeg", ".png", ".webp"}
OWN_OUTPUT = {"lora_output", "smoke_test"}  # an attached earlier output is not a dataset


def images_in(folder: Path) -> list[Path]:
    return sorted(path for path in folder.iterdir()
                  if path.is_file() and path.suffix.lower() in IMAGE_TYPES)


def caption_for(image: Path) -> str | None:
    for path in image.parent.iterdir():
        if path.suffix.lower() == ".txt" and path.stem.lower() == image.stem.lower():
            return path.read_text(encoding="utf-8-sig", errors="ignore").strip() or None
    return None


for packed in Path("/kaggle/input").rglob("*.zip"):
    if packed.name.endswith("_outputs.zip"):
        continue  # an earlier training output, not the dataset
    target = Path(SCRATCH, "unzipped", packed.stem)
    if not target.exists():
        try:
            with zipfile.ZipFile(packed) as bundle:
                bundle.extractall(target)
        except zipfile.BadZipFile:
            print("skipped (not a zip):", packed)
roots = [Path("/kaggle/input"), Path(SCRATCH, "unzipped")]
folders = [folder for root in roots if root.exists() for folder in [root, *root.rglob("*")]
           if folder.is_dir() and not OWN_OUTPUT & set(folder.parts)]
found = max(folders, key=lambda folder: sum(caption_for(image) is not None
                                             for image in images_in(folder)), default=None)
images = images_in(found) if found is not None else []
captions = {image: caption_for(image) for image in images}
firsts = Counter(text.split(",")[0].strip() for text in captions.values() if text)
if len(images) < MIN_IMAGES or not firsts:
    fail(f"no dataset with at least {MIN_IMAGES} captioned images: attach the private dataset "
         "made by `ca lora prepare` (Add Input)")
TRIGGER = firsts.most_common(1)[0][0]
DATASET = Path(SCRATCH, "dataset")
shutil.rmtree(DATASET, ignore_errors=True)
DATASET.mkdir(parents=True)
fixed = []
for number, image in enumerate(images, start=1):
    text = captions[image] or f"{TRIGGER}, a man"
    if not text.lower().startswith(TRIGGER.lower()):
        text = f"{TRIGGER}, {text}"
    if text != captions[image]:
        fixed.append(image.name)
    stem = f"{number:02d}"
    shutil.copy(image, DATASET / f"{stem}{image.suffix.lower()}")
    Path(DATASET, f"{stem}.txt").write_text(text + "\n", encoding="utf-8")
JOB = f"{TRIGGER}_z_image_turbo"
print(f"Dataset: {found} -> {len(images)} images, trigger {TRIGGER!r}, job {JOB}")
if fixed:
    print("captions added or fixed for:", ", ".join(fixed))
save_status(dataset=str(found), images=len(images), trigger=TRIGGER, job=JOB,
            captions_fixed=fixed)

In [ ]:
# AI Toolkit at the pinned commit, then its requirements in ONE pip run with every version
# pinned. Kaggle's torch is held by a constraints file, so pip fails instead of replacing it.
import re
from importlib.metadata import PackageNotFoundError, version

TOOLKIT = Path(SCRATCH, "ai-toolkit")
if not (TOOLKIT / "run.py").exists():
    TOOLKIT.mkdir(parents=True, exist_ok=True)
    sh(f"git -C {TOOLKIT} init -q && git -C {TOOLKIT} remote add origin {TOOLKIT_REPO} && "
       f"git -C {TOOLKIT} fetch -q --depth 1 origin {TOOLKIT_COMMIT} && "
       f"git -C {TOOLKIT} checkout -q FETCH_HEAD")
commit = subprocess.run(["git", "-C", str(TOOLKIT), "rev-parse", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
if commit != TOOLKIT_COMMIT:
    fail(f"AI Toolkit is at {commit or 'nothing'}, not the pinned {TOOLKIT_COMMIT}")


def installed(name: str) -> str | None:
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def requirement_lines(path: Path) -> list[str]:
    lines: list[str] = []
    for raw in path.read_text().splitlines():
        line = raw.split(" #", 1)[0].strip()
        if not line or line.startswith("#"):
            continue
        include = re.match(r"^(?:-r|--requirement)\s*=?\s*(\S+)", line)
        if include:
            lines += requirement_lines(path.parent / include.group(1))
        elif not line.startswith("-"):
            lines.append(line)
    return lines


specs: list[str] = []
unpinned: list[str] = []
for line in requirement_lines(TOOLKIT / "requirements.txt"):
    if line.startswith("git+"):
        specs.append(line)  # a source pinned to a commit by AI Toolkit itself
        continue
    plain = re.match(r"^([A-Za-z0-9][A-Za-z0-9._-]*)", line)
    name = plain.group(1).lower().replace("_", "-") if plain else line
    if name in SKIP_PACKAGES:
        continue
    if name not in PINNED:
        unpinned.append(line)
    else:
        specs.append(PINNED[name])
if unpinned:
    fail(f"requirements without a pin (add them to PINNED): {unpinned}")
specs += EXTRA_PINS
Path(SCRATCH, "pinned.txt").write_text("\n".join(specs) + "\n")
held = [f"{name}=={installed(name)}" for name in ("torch", "torchvision", "torchaudio")
        if installed(name)]
Path(SCRATCH, "constraints.txt").write_text("\n".join(held) + "\n")
print("holding:", held)
if installed("bitsandbytes"):
    sh(f"{sys.executable} -m pip uninstall -y -q bitsandbytes")
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", f"{SCRATCH}/pinned.txt",
     "-c", f"{SCRATCH}/constraints.txt"], capture_output=True, text=True)
if install.returncode != 0:
    fail(f"pip could not install the pinned set: {(install.stdout + install.stderr)[-800:]}")
if installed("bitsandbytes"):
    fail("bitsandbytes came back as a dependency; it must not be installed")
VERSIONS = {name: installed(name) or "-" for name in (
    "torch", "torchao", "optimum-quanto", "diffusers", "transformers", "accelerate",
    "safetensors", "peft", "numpy", "scipy", "opencv-python")}
VERSIONS["python"] = sys.version.split()[0]
print("versions:", VERSIONS)
save_status(toolkit_commit=commit, versions=VERSIONS)

In [ ]:
# Z-Image as this AI Toolkit commit registers it, and the pinned Turbo training adapter.
from huggingface_hub import list_repo_files

if 'arch = "zimage"' not in (TOOLKIT / "extensions_built_in/diffusion_models/z_image/"
                             "z_image.py").read_text(errors="ignore"):
    fail("this AI Toolkit commit does not register the zimage arch")
ARCH = "zimage"
if ADAPTER_FILE not in list_repo_files(ADAPTER_REPO):
    fail(f"{ADAPTER_FILE} is not in {ADAPTER_REPO}")
ADAPTER = f"{ADAPTER_REPO}/{ADAPTER_FILE}"
print("Arch:", ARCH, "| adapter:", ADAPTER)
save_status(arch=ARCH, adapter=ADAPTER)

In [ ]:
# Resume. AI Toolkit continues from the checkpoint file in the job folder with the NEWEST
# CREATION TIME (not the highest step). So the job folder must hold the chosen checkpoint as
# its newest file: the newest one that learned something (finite weights, LoRA up/B weights
# not all zero), from /kaggle/working or an attached earlier output. Anything newer or unusable
# moves out of the folder, and optimizer.pt only stays when it belongs to that checkpoint.
from safetensors.torch import load_file

JOB_DIR = Path(OUTPUT_DIR, JOB)
ASIDE = Path(OUTPUT_DIR, f"{JOB}_set_aside")
STEP_PATTERN = re.compile(r"_(\d+)\.safetensors$")


def step_of(path: Path) -> int:
    match = STEP_PATTERN.search(path.name)
    return int(match.group(1)) if match else 10**9  # the final file: after every step


def learned(path: Path) -> bool:
    try:
        state = load_file(str(path))
    except Exception as error:  # noqa: BLE001 - a broken file is simply not usable
        print("unreadable:", path, error)
        return False
    if not all(bool(torch.isfinite(tensor.float()).all()) for tensor in state.values()):
        return False
    ups = [tensor for key, tensor in state.items() if "lora_B" in key or "lora_up" in key]
    return not ups or any(bool(tensor.abs().max() > 0) for tensor in ups)


def move_aside(path: Path) -> None:
    ASIDE.mkdir(parents=True, exist_ok=True)
    shutil.move(str(path), str(ASIDE / f"{int(time.time())}_{path.name}"))


sources = [JOB_DIR] + [folder for folder in Path("/kaggle/input").rglob(JOB)
                       if folder.is_dir() and "smoke_test" not in folder.parts]
saved = sorted((path for folder in sources if folder.exists()
                for path in folder.glob("*.safetensors")), key=step_of, reverse=True)
RESUMED_FROM = next((path for path in saved if learned(path)), None)
for path in list(JOB_DIR.glob("*.safetensors")) if JOB_DIR.exists() else []:
    if RESUMED_FROM is None or step_of(path) > step_of(RESUMED_FROM):
        move_aside(path)  # newer than the chosen one, or nothing usable at all
newest_in_source = RESUMED_FROM is not None and step_of(RESUMED_FROM) == max(
    step_of(path) for path in RESUMED_FROM.parent.glob("*.safetensors"))
optimizer_state = JOB_DIR / "optimizer.pt"
if optimizer_state.exists() and (RESUMED_FROM is None or not newest_in_source
                                 or RESUMED_FROM.parent != JOB_DIR):
    move_aside(optimizer_state)  # it belongs to another state
if RESUMED_FROM is not None and RESUMED_FROM.parent != JOB_DIR:
    JOB_DIR.mkdir(parents=True, exist_ok=True)
    if (RESUMED_FROM.parent / "samples").is_dir():
        shutil.copytree(RESUMED_FROM.parent / "samples", JOB_DIR / "samples", dirs_exist_ok=True)
    if newest_in_source and (RESUMED_FROM.parent / "optimizer.pt").exists():
        shutil.copy(RESUMED_FROM.parent / "optimizer.pt", optimizer_state)
    target = JOB_DIR / RESUMED_FROM.name
    shutil.copy(RESUMED_FROM, target)  # copied last: the newest file, so AI Toolkit picks it
    os.utime(target)
print("Resume from:", RESUMED_FROM or "nothing usable: a fresh start")
save_status(resumed_from=str(RESUMED_FROM) if RESUMED_FROM else None)

In [ ]:
# The AI Toolkit config (YAML; JSON strings and numbers are valid YAML scalars). Every key is
# read by AI Toolkit's config classes at TOOLKIT_COMMIT. model.dtype and save.dtype default to
# float16 there, so both are set: the model, text encoder and VAE load in bf16.


def q(value: object) -> str:
    return json.dumps(value)


def write_config(path: Path, name: str, folder: str, *, steps: int, resolutions: list[int],
                 sample_size: int, prompts: list[str], every: int) -> None:
    listed = "\n".join(f"          - {q(prompt)}" for prompt in prompts)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(f"""job: extension
config:
  name: {q(name)}
  process:
    - type: "sd_trainer"
      training_folder: {q(folder)}
      device: "cuda:0"
      trigger_word: {q(TRIGGER)}
      network:
        type: "lora"
        linear: {RANK}
        linear_alpha: {RANK}
      save:
        dtype: {q(DTYPE)}
        save_every: {every}
        max_step_saves_to_keep: {KEEP_CHECKPOINTS}
      datasets:
        - folder_path: {q(str(DATASET))}
          caption_ext: "txt"
          caption_dropout_rate: 0.05
          cache_latents_to_disk: true
          resolution: {q(resolutions)}
      train:
        batch_size: 1
        cache_text_embeddings: true
        skip_first_sample: true
        steps: {steps}
        train_unet: true
        train_text_encoder: false
        gradient_checkpointing: true
        noise_scheduler: "flowmatch"
        optimizer: {q(OPTIMIZER)}
        lr: {LEARNING_RATE}
        dtype: {q(DTYPE)}
      model:
        name_or_path: {q(BASE_MODEL)}
        arch: {q(ARCH)}
        dtype: {q(DTYPE)}
        assistant_lora_path: {q(ADAPTER)}
        quantize: true
        qtype: "qfloat8"
        quantize_te: true
        qtype_te: "qfloat8"
        low_vram: true
      sample:
        sampler: "flowmatch"
        sample_every: {every}
        width: {sample_size}
        height: {sample_size}
        prompts:
{listed}
        neg: ""
        seed: 42
        walk_seed: true
        guidance_scale: 1
        sample_steps: 8
meta:
  name: "[name]"
  version: "1.0"
""")


CONFIG = Path(OUTPUT_DIR, f"{JOB}.yaml")
SMOKE_NAME = f"{JOB}_smoke"
SMOKE_CONFIG = Path(SMOKE_DIR, f"{SMOKE_NAME}.yaml")
write_config(CONFIG, JOB, OUTPUT_DIR, steps=STEPS, resolutions=RESOLUTIONS,
             sample_size=SAMPLE_SIZE, prompts=SAMPLE_PROMPTS, every=SAVE_EVERY)
print(CONFIG.read_text())

In [ ]:
# Shared by the smoke test and the full run: progress parsing, memory, stopping.
import math
import signal

PROGRESS = re.compile(r"(\d+)/(\d+) \[[^\]]*?loss:\s*([-+0-9.eE]+|nan|inf)", re.IGNORECASE)


def losses(text: str) -> dict[int, str]:
    return {int(step): value.lower() for step, _, value in PROGRESS.findall(text)}


def bad(value: str) -> bool:
    return value in ("nan", "inf", "-inf") or not math.isfinite(float(value)) or (
        float(value) == 0.0)


def memory() -> str:
    info = dict(line.split(":", 1) for line in Path("/proc/meminfo").read_text().splitlines())
    free = int(info["MemAvailable"].split()[0]) / 2**20
    total = int(info["MemTotal"].split()[0]) / 2**20
    gpu = subprocess.run(["nvidia-smi", "--id=0", "--query-gpu=memory.used,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    return f"RAM free {free:.1f}/{total:.1f} GB | GPU 0 {gpu or 'n/a'}"


def stop(process: subprocess.Popen[bytes], wait_s: int) -> None:
    os.killpg(process.pid, signal.SIGINT)  # AI Toolkit stops at Ctrl+C
    try:
        process.wait(timeout=wait_s)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)


def last_error(text: str) -> str:
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    hits = [line for line in lines if re.search(r"error|exception|fatal|segmentation|killed",
                                                line, re.IGNORECASE)]
    return (hits or lines or ["no output"])[-1][-300:]

In [ ]:
# SMOKE TEST (<= 15 min): imports -> AI Toolkit's own config loader -> model on GPU 0 ->
# 3 real steps with a real loss -> checkpoint saved -> reloaded (finite, learned, step stored
# for resume) -> 1 sample at 512 px (exists, not blank). Any failure stops the notebook here.
from PIL import Image, ImageStat
from safetensors import safe_open

CHECK = r"""
import importlib.util, json, sys
from toolkit.job import get_job
import jobs.ExtensionJob, extensions_built_in.sd_trainer.SDTrainer
import extensions_built_in.diffusion_models.z_image.z_image, toolkit.util.quantize
import toolkit.optimizer, optimum.quanto, torchao, diffusers, transformers, peft, accelerate
assert importlib.util.find_spec("bitsandbytes") is None, "bitsandbytes is installed"
process = get_job(sys.argv[1]).process[0]
print(json.dumps({"process": type(process).__name__, "train_dtype": process.train_config.dtype,
    "model_dtype": process.model_config.dtype, "te_dtype": process.model_config.te_dtype,
    "vae_dtype": process.model_config.vae_dtype, "save_dtype": process.save_config.dtype,
    "arch": process.model_config.arch, "optimizer": process.train_config.optimizer,
    "qtype": process.model_config.qtype}))
"""
EXPECTED = {"process": "SDTrainer", "train_dtype": "bf16", "model_dtype": "bf16",
            "te_dtype": "bf16", "vae_dtype": "bf16", "save_dtype": "bf16", "arch": "zimage",
            "optimizer": "adamw", "qtype": "qfloat8"}


def smoke_test() -> str | None:
    """None when everything works, else the exact reason."""
    deadline = time.time() + SMOKE_MINUTES * 60
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)
    write_config(SMOKE_CONFIG, SMOKE_NAME, SMOKE_DIR, steps=SMOKE_STEPS,
                 resolutions=[SMOKE_RESOLUTION], sample_size=SMOKE_SAMPLE_SIZE,
                 prompts=SAMPLE_PROMPTS[:1], every=1000)
    try:
        check = subprocess.run([sys.executable, "-c", CHECK, str(SMOKE_CONFIG)], cwd=TOOLKIT,
                               env=TRAIN_ENV, capture_output=True, text=True, timeout=600)
    except subprocess.TimeoutExpired:
        return "the import and config check took longer than 10 minutes"
    if check.returncode != 0:
        return f"import/config check failed: {last_error(check.stdout + check.stderr)}"
    try:
        parsed = json.loads(check.stdout.strip().splitlines()[-1])
    except (IndexError, ValueError):
        return f"the config check printed no result: {last_error(check.stdout + check.stderr)}"
    wrong = {key: parsed.get(key) for key, value in EXPECTED.items() if parsed.get(key) != value}
    if wrong:
        return f"AI Toolkit parsed the config differently than written: {wrong}"
    log_path = Path(SMOKE_DIR, "smoke_log.txt")
    with log_path.open("w") as log:
        process = subprocess.Popen([sys.executable, "run.py", str(SMOKE_CONFIG)], cwd=TOOLKIT,
                                   stdout=log, stderr=subprocess.STDOUT, env=TRAIN_ENV,
                                   start_new_session=True)
        try:
            process.wait(timeout=max(60, deadline - time.time()))
        except subprocess.TimeoutExpired:
            stop(process, 60)
            return f"the smoke run did not finish within {SMOKE_MINUTES} minutes"
    text = log_path.read_text(errors="ignore")
    if process.returncode != 0:
        if "out of memory" in text.lower():
            return (f"out of GPU memory at {SMOKE_RESOLUTION} px: set RESOLUTIONS = [512] and "
                    "SMOKE_RESOLUTION = 512")
        return f"the trainer exited with code {process.returncode}: {last_error(text)}"
    seen = losses(text)
    if len(seen) < 2 or any(bad(value) for value in seen.values()) or "loss is nan" in text:
        return f"no real loss in the smoke steps: {seen or 'no progress lines'}"
    files = sorted(Path(SMOKE_DIR, SMOKE_NAME).glob("*.safetensors"),
                   key=lambda path: path.stat().st_mtime)
    if not files:
        return "the smoke run saved no checkpoint"
    if not learned(files[-1]):
        return f"the smoke checkpoint {files[-1].name} has NaN or untrained weights"
    with safe_open(str(files[-1]), framework="pt") as handle:
        meta = handle.metadata() or {}
    try:
        step = int(json.loads(meta.get("training_info", "{}")).get("step", 0))
    except (ValueError, TypeError):
        step = 0
    if step < 1:
        return "the smoke checkpoint stores no training step, so resuming would restart"
    samples = sorted(Path(SMOKE_DIR, SMOKE_NAME, "samples").glob("*.jpg"))
    if not samples:
        return "the smoke run made no sample image"
    with Image.open(samples[-1]) as image:
        spread = max(ImageStat.Stat(image.convert("RGB")).stddev)
        size = image.size
    if spread < 5:
        return f"the smoke sample {samples[-1].name} is blank (NaN in sampling?)"
    print(f"Smoke test passed in {(time.time() - deadline) / 60 + SMOKE_MINUTES:.1f} min: "
          f"losses {seen}, checkpoint {files[-1].name} (step {step}), sample {size}")
    return None


reason = smoke_test()
save_status(smoke_test="passed" if reason is None else f"failed: {reason}")
if reason is not None:
    fail(f"smoke test: {reason} (see smoke_test/smoke_log.txt)")

In [ ]:
# The full run. Never raises: whatever happens, the next cell packs every finished checkpoint.
# Every PROGRESS_EVERY steps one line (notebook + train_log.txt): step, loss, s/step, ETA,
# free RAM, GPU memory, last checkpoint. The guard stops a run whose loss is 0/NaN/inf for the
# first GUARD_STEPS steps; the run also stops before the session limit.
LOG = Path(OUTPUT_DIR, "train_log.txt")
DEADLINE = NOTEBOOK_START + SESSION_HOURS * 3600 - END_MARGIN_MINUTES * 60


def checkpoints() -> list[Path]:
    return sorted(JOB_DIR.glob("*.safetensors"), key=step_of)


def loss_problem(seen: dict[int, str]) -> str | None:
    if len(seen) < 2 or max(seen) - min(seen) < GUARD_STEPS:
        return None
    first = [value for step, value in sorted(seen.items()) if step < min(seen) + GUARD_STEPS]
    if all(bad(value) for value in first):
        return (f"loss was 0/NaN for the first {GUARD_STEPS} steps (dtype {DTYPE}): nothing "
                "would be learned")
    return None


def progress(seen: dict[int, str], times: list[tuple[int, float]]) -> str:
    step = max(seen)
    window = [(s, t) for s, t in times if s >= step - 40] or times
    rate = ((window[-1][1] - window[0][1]) / (window[-1][0] - window[0][0])
            if len(window) > 1 and window[-1][0] > window[0][0] else float("nan"))
    eta = (STEPS - step) * rate
    eta_text = "n/a" if math.isnan(eta) else f"{int(eta // 3600)}h{int(eta % 3600 // 60):02d}"
    last = checkpoints()[-1].name if checkpoints() else "none yet"
    return (f"[progress] step {step}/{STEPS} | loss {seen[step]} | {rate:.1f} s/step | "
            f"ETA {eta_text} | {memory()} | last checkpoint {last}")


def run_training(resolutions: list[int]) -> str:
    write_config(CONFIG, JOB, OUTPUT_DIR, steps=STEPS, resolutions=resolutions,
                 sample_size=SAMPLE_SIZE, prompts=SAMPLE_PROMPTS, every=SAVE_EVERY)
    with LOG.open("a") as log:
        log.write(f"\n=== training at {resolutions}, dtype {DTYPE}, versions {VERSIONS} ===\n")
        log.flush()
        offset = LOG.stat().st_size  # this run's part of the log only
        process = subprocess.Popen([sys.executable, "run.py", str(CONFIG)], cwd=TOOLKIT,
                                   stdout=log, stderr=subprocess.STDOUT, env=TRAIN_ENV,
                                   start_new_session=True)
        times: list[tuple[int, float]] = []
        reported = -1
        while process.poll() is None:
            time.sleep(20)
            with LOG.open("rb") as reader:
                reader.seek(offset)
                text = reader.read().decode(errors="ignore")
            seen = losses(text)
            if seen and (not times or max(seen) > times[-1][0]):
                times.append((max(seen), time.time()))
            problem = loss_problem(seen)
            if problem:
                stop(process, 60)
                return f"failed: {problem}"
            if time.time() > DEADLINE:
                stop(process, 300)
                return "stopped: session time limit reached (resume in the next session)"
            if seen and max(seen) // PROGRESS_EVERY > reported // PROGRESS_EVERY:
                reported = max(seen)
                line = progress(seen, times)
                print(line, flush=True)
                log.write(f"\n{line}\n")
                log.flush()
    with LOG.open("rb") as reader:
        reader.seek(offset)
        text = reader.read().decode(errors="ignore")
    if process.returncode == 0:
        return "done"
    if "out of memory" in text.lower() and not checkpoints():
        return "oom"
    return f"failed: exit code {process.returncode}: {last_error(text)}"


try:
    result = run_training(RESOLUTIONS)
    if result == "oom":
        print("Out of GPU memory before the first checkpoint: trying once at", FALLBACK_RESOLUTIONS)
        STATUS["notes"] = [f"out of memory at {RESOLUTIONS}: trained at {FALLBACK_RESOLUTIONS}"]
        result = run_training(FALLBACK_RESOLUTIONS)
        if result == "oom":
            result = f"failed: out of GPU memory also at {FALLBACK_RESOLUTIONS}"
except Exception as error:  # noqa: BLE001 - the outputs are packed whatever happened
    result = f"failed: {type(error).__name__}: {error}"
save_status(status=result, checkpoints=[path.name for path in checkpoints()])
print("Training:", result)

In [ ]:
# Pack every checkpoint, sample, the config, the logs and status.json into one zip.
ARCHIVE = Path(WORK, f"{JOB}_outputs.zip")
with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(Path(OUTPUT_DIR).rglob("*")):
        if path.is_file() and "_latent_cache" not in path.parts:
            bundle.write(path, path.relative_to(OUTPUT_DIR))
    smoke_log = Path(SMOKE_DIR, "smoke_log.txt")
    if smoke_log.exists():
        bundle.write(smoke_log, "smoke_log.txt")
samples = sorted(Path(JOB_DIR, "samples").glob("*"))
print(f"Checkpoints ({len(checkpoints())}):", *[path.name for path in checkpoints()], sep="\n  ")
print(f"Sample images: {len(samples)} in {Path(JOB_DIR, 'samples')}")
print(f"Everything in one file: {ARCHIVE} ({ARCHIVE.stat().st_size / 2**20:.0f} MB)")
print(f"Status: {STATUS['status']}")